# 03 — Regression Analysis: OD vs SD

Fit a standard multiple linear regression model on each pair of **Original Data (OD)** and **Synthetic Data (SD)** datasets.  
Compare estimated coefficients side-by-side and persist the fitted models for evaluation in Step 04.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json
import os
import glob
import pickle

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
# ── Load Configuration ──────────────────────────────────────────────────────
config_path = os.path.join("..", "config", "config.json")
with open(config_path, "r") as f:
    config = json.load(f)

print("Configuration loaded:")
print(json.dumps(config, indent=2))

In [ ]:
# ── Discover matched OD / SD file pairs ────────────────────────────────────
od_dir = os.path.join("..", "data", "original")
sd_dir = os.path.join("..", "data", "synthetic")
model_dir = os.path.join("..", "models")
os.makedirs(model_dir, exist_ok=True)

od_files = sorted(glob.glob(os.path.join(od_dir, "OD_*.csv")))
sd_files = sorted(glob.glob(os.path.join(sd_dir, "SD_*.csv")))

assert len(od_files) > 0, f"No OD files found in {od_dir}"
assert len(od_files) == len(sd_files), (
    f"Mismatch: {len(od_files)} OD vs {len(sd_files)} SD files"
)

print(f"Found {len(od_files)} OD/SD pair(s):")
for f in od_files:
    print(f"  {os.path.basename(f)}")

## Fit OLS Models

For each scenario (one per `rho` value), fit $y = X\beta + \epsilon$ on both the OD and SD datasets using ordinary least squares.

In [ ]:
# ── Fit models for each OD / SD pair ───────────────────────────────────────
results = []

for od_path, sd_path in zip(od_files, sd_files):
    # Read data
    od = pd.read_csv(od_path)
    sd = pd.read_csv(sd_path)

    scenario = os.path.basename(od_path).replace("OD_", "").replace(".csv", "")
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario}")
    print(f"{'='*60}")

    # Separate features & target
    X_od, y_od = od.drop(columns="y"), od["y"]
    X_sd, y_sd = sd.drop(columns="y"), sd["y"]

    # Fit OLS on Original Data
    lm_od = LinearRegression().fit(X_od, y_od)
    r2_od = r2_score(y_od, lm_od.predict(X_od))

    # Fit OLS on Synthetic Data
    lm_sd = LinearRegression().fit(X_sd, y_sd)
    r2_sd = r2_score(y_sd, lm_sd.predict(X_sd))

    print(f"  OD model R²: {r2_od:.6f}")
    print(f"  SD model R²: {r2_sd:.6f}")

    # Build coefficient comparison table
    terms = ["(Intercept)"] + [f"X{i+1}" for i in range(X_od.shape[1])]
    beta_od = np.concatenate([[lm_od.intercept_], lm_od.coef_])
    beta_sd = np.concatenate([[lm_sd.intercept_], lm_sd.coef_])

    coef_df = pd.DataFrame({
        "term": terms,
        "beta_OD": np.round(beta_od, 6),
        "beta_SD": np.round(beta_sd, 6),
        "diff": np.round(beta_od - beta_sd, 6),
    })
    display(coef_df)

    # Save paired models
    model_path = os.path.join(model_dir, f"models_{scenario}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump({"lm_od": lm_od, "lm_sd": lm_sd, "scenario": scenario}, f)
    print(f"  Saved: {os.path.basename(model_path)}")

    results.append({
        "scenario": scenario,
        "r2_od": r2_od,
        "r2_sd": r2_sd,
    })

## Summary

In [ ]:
summary_df = pd.DataFrame(results)
display(summary_df)
print("\n[DONE] Regression analysis complete. Models saved to ../models/")